In [28]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU detected: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [3]:
X.shape

torch.Size([2, 3])

In [5]:
X.dtype

torch.float32

In [6]:
X[0, 1]

tensor(4.)

In [7]:
X[1, 2]

tensor(6.)

In [8]:
X[:, 1]

tensor([4., 3.])

In [10]:
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [9]:
10 * (X + 1.0)  # itemwise addition and multiplication

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [11]:
X.exp()  # itemwise exponential

tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])

In [12]:
X.mean()

tensor(3.8333)

In [13]:
X.max(dim=0)

torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))

In [14]:
X @ X.T  # matrix transpose and matrix multiplication

tensor([[66., 56.],
        [56., 49.]])

In [15]:
import numpy as np
X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [16]:
torch.tensor(np.array([[1., 2., 7.], [2., 3., 6.]]))

tensor([[1., 2., 7.],
        [2., 3., 6.]], dtype=torch.float64)

In [ ]:
torch.FloatTensor(np.array([[1., 4., 7.], [2., 3., 6]])) #converts to 32 bits as it is preferred for faster computation

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [18]:
X[:, 1] = -99
X

tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In [20]:
X.relu_()
X

tensor([[1., 0., 7.],
        [2., 0., 6.]])

In [29]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

'cuda'

In [25]:
M = torch.tensor([[1., 2., 3.,], [4., 5., 6.]])
M = M.to(device)

In [26]:
M.device

device(type='cuda', index=0)

In [27]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

In [28]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

In [29]:
M = torch.rand((1000, 1000)) #on the cpu
%timeit M @ M.T

3.51 ms ± 47.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [32]:
M = torch.rand((1000, 1000), device="cuda")  # on the GPU
%timeit M @ M.T

476 μs ± 7.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


 For example,consider a simple function, f(x) = x2. Differential calculus tells us that the derivative of this function is f’(x) = 2x. If we evaluate f(5) and f'(5), we get 25 and 10, respectively. Let’s see if PyTorch agrees:   [pg no 323, 324]

In [35]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [36]:
f.backward()
x.grad

tensor(10.)

In [37]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad  #gradient descent step

In [38]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad

In [39]:
x.grad.zero_()

tensor(0.)

In [40]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2  # forward pass
    f.backward()  # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad  # gradient descent step
        x.grad.zero_()  # reset the gradients

In [ ]:
# t = torch.tensor(2.0, requires_grad=True)
# z = t.exp()  # this is an intermediate result
# z += 1  # this is an in-place operation
# z.backward()  #  RuntimeError!

### Autograd

That is the exact right question to ask. You know why the math results in 10, but how does typing the specific word .backward() actually make the computer calculate it?

Here is exactly what happens under the hood when you trigger that function.

1. The Hidden Flowchart (The Forward Pass)
When you ran f = x ** 2, PyTorch didn't just calculate 25 and call it a day. Because you told it to keep an eye on x (requires_grad=True), it built a mini-flowchart in your computer's memory.

This flowchart remembers two crucial pieces of information:

The Input: The specific number 5.0.

The Operation: The "power of 2" function.

2. The Built-in Cheat Sheet
PyTorch doesn't actually "think" or do calculus from scratch. Instead, it has a massive rulebook programmed into its core code.

Its developers already hardcoded the calculus shortcut we talked about earlier: Whenever you see a "power of 2" operation, the derivative formula is "2 times the input".

3. Executing .backward()
When you type f.backward(), you are telling PyTorch to wake up, grab its rulebook, and walk backwards through the flowchart it just built.

Here is PyTorch's internal process when you run that single line of code:

"Okay, the user called .backward() on f. I need to work my way back to the starting variable, x."

"Looking at the flowchart... to create f, I used a 'power of 2' operation."

"Let me check my calculus cheat sheet for the 'power of 2' rule. Ah, the rule is 2 * input."

"Looking at my flowchart again, the input was 5.0. Let's do the math: 2 * 5.0 = 10.0."

"I will now take this 10.0, walk all the way back to the original x variable, and lock this number inside a special storage container attached to x, called .grad."

4. The Reveal
Notice that the .backward() function doesn't actually print anything to your screen. It does all of that flowchart-walking and math entirely in the background.

That is why you have to write x.grad on the very last line. You are essentially saying, "Hey PyTorch, open up that .grad storage container for x and show me the number you put in there during your backward walk."

And that is exactly when it hands you the 10.

In [ ]:
# pg 326

### https://0byte.io/articles/pytorch_introduction.html

### Linear Regression Using Tensors and Autograd

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch

housing = fetch_california_housing()
X, y = housing.data, housing.target

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42
)

X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)

means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)



In [4]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [15]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs},  Loss: {loss.item()}")

Epoch 1/20,  Loss: 0.5153099894523621
Epoch 2/20,  Loss: 0.5152389407157898
Epoch 3/20,  Loss: 0.5151732563972473
Epoch 4/20,  Loss: 0.5151126384735107
Epoch 5/20,  Loss: 0.5150566697120667
Epoch 6/20,  Loss: 0.5150050520896912
Epoch 7/20,  Loss: 0.5149574279785156
Epoch 8/20,  Loss: 0.5149133205413818
Epoch 9/20,  Loss: 0.514872670173645
Epoch 10/20,  Loss: 0.5148350596427917
Epoch 11/20,  Loss: 0.5148003697395325
Epoch 12/20,  Loss: 0.5147683024406433
Epoch 13/20,  Loss: 0.5147386789321899
Epoch 14/20,  Loss: 0.5147113800048828
Epoch 15/20,  Loss: 0.514686107635498
Epoch 16/20,  Loss: 0.5146628022193909
Epoch 17/20,  Loss: 0.5146411657333374
Epoch 18/20,  Loss: 0.5146213173866272
Epoch 19/20,  Loss: 0.5146028995513916
Epoch 20/20,  Loss: 0.5145859718322754


### Linear Regression Using PyTorch’s High-Level API

In [5]:
import torch.nn as nn
torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [6]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [7]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [8]:
model(X_train[:2])

tensor([[0.4296],
        [1.1455]], grad_fn=<AddmmBackward0>)

In [ ]:
learning_rate = 0.4
n_epochs = 20

In [10]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [15]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [17]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.272577285766602
Epoch 2/20, Loss: 0.7673617601394653
Epoch 3/20, Loss: 0.6151816844940186
Epoch 4/20, Loss: 0.5950986742973328
Epoch 5/20, Loss: 0.5839646458625793
Epoch 6/20, Loss: 0.5752211213111877
Epoch 7/20, Loss: 0.567899227142334
Epoch 8/20, Loss: 0.5616247057914734
Epoch 9/20, Loss: 0.5561909079551697
Epoch 10/20, Loss: 0.5514596104621887
Epoch 11/20, Loss: 0.5473267436027527
Epoch 12/20, Loss: 0.5437079668045044
Epoch 13/20, Loss: 0.5405329465866089
Epoch 14/20, Loss: 0.5377423763275146
Epoch 15/20, Loss: 0.5352851748466492
Epoch 16/20, Loss: 0.5331176519393921
Epoch 17/20, Loss: 0.5312021970748901
Epoch 18/20, Loss: 0.529506504535675
Epoch 19/20, Loss: 0.5280026197433472
Epoch 20/20, Loss: 0.5266665816307068


In [18]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = model(X_new)
y_pred

tensor([[0.8226],
        [1.6903],
        [2.6812]])

### Implementing a Regression MLP

In [23]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [24]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.974284648895264
Epoch 2/20, Loss: 2.04504132270813
Epoch 3/20, Loss: 1.0027456283569336
Epoch 4/20, Loss: 0.8672141432762146
Epoch 5/20, Loss: 0.7867145538330078
Epoch 6/20, Loss: 0.7360551953315735
Epoch 7/20, Loss: 0.7031640410423279
Epoch 8/20, Loss: 0.6811584234237671
Epoch 9/20, Loss: 0.6656913757324219
Epoch 10/20, Loss: 0.6541036367416382
Epoch 11/20, Loss: 0.6448375582695007
Epoch 12/20, Loss: 0.6369345784187317
Epoch 13/20, Loss: 0.6298699975013733
Epoch 14/20, Loss: 0.6233502626419067
Epoch 15/20, Loss: 0.6172211766242981
Epoch 16/20, Loss: 0.6113643050193787
Epoch 17/20, Loss: 0.6057479381561279
Epoch 18/20, Loss: 0.6003261208534241
Epoch 19/20, Loss: 0.5950582027435303
Epoch 20/20, Loss: 0.5899456143379211


In [25]:
from torch.utils.data import TensorDataset,  DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [30]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)
model = model.to(device)

In [35]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [36]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 4.9743
Epoch 2/20, Loss: 4.9743
Epoch 3/20, Loss: 4.9743
Epoch 4/20, Loss: 4.9743
Epoch 5/20, Loss: 4.9743
Epoch 6/20, Loss: 4.9743
Epoch 7/20, Loss: 4.9743
Epoch 8/20, Loss: 4.9743
Epoch 9/20, Loss: 4.9743
Epoch 10/20, Loss: 4.9743
Epoch 11/20, Loss: 4.9743
Epoch 12/20, Loss: 4.9743
Epoch 13/20, Loss: 4.9743
Epoch 14/20, Loss: 4.9743
Epoch 15/20, Loss: 4.9743
Epoch 16/20, Loss: 4.9743
Epoch 17/20, Loss: 4.9743
Epoch 18/20, Loss: 4.9743
Epoch 19/20, Loss: 4.9743
Epoch 20/20, Loss: 4.9743
